# kin_04 — Slow kinematics timecourse

Does movement vigor change systematically across the session?
Are post-rewarded vs post-unrewarded trials different in vigor?

**Pipeline:**
1. Load `all_tongue_movements` parquet (session-keyed, no spike data)
2. Single-session: RT moving average over trials
3. Population: z-score within session → 100-trial MA → interpolate to 0–1 fraction → grand mean ± SEM
4. First-movement vs all-movements comparison
5. Post-rewarded vs post-unrewarded conditioning effect

## 1. Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "kin_04_timecourse"
SAVE_FIG = False
print(f"ENV={ENV}")

## 2. Load data

In [ ]:
if ENV == "codeocean":
    movements_path = SCRATCH / "all_tongue_movements_04022026" / "all_tongue_movements_04022026.parquet"
else:
    movements_path = FOR_LOCAL / "all_tongue_movements_04022026.parquet"

all_tongue_movements = pd.read_parquet(movements_path)
print("Shape:", all_tongue_movements.shape)
print("Sessions:", all_tongue_movements["session"].nunique())
print("Columns:", list(all_tongue_movements.columns))

## 3. Single-session RT timecourse

Plot reaction time (movement latency from go) as a function of trial number
with 50- and 100-trial moving averages, for one example session.

In [ ]:
RT_COL = "movement_latency_from_go"
VIGOR_COL = "mean_velocity"  # primary vigor measure

# Choose one example session
example_session = all_tongue_movements["session"].unique()[0]
print("Example session:", example_session)

sess_df = all_tongue_movements[all_tongue_movements["session"] == example_session].copy()

# One cue-response movement per trial (earliest in trial)
first_movs = (
    sess_df
    .sort_values(["trial", "start_time"])
    .groupby("trial", as_index=False)
    .first()
    .sort_values("trial")
)

first_movs = first_movs.dropna(subset=[RT_COL])
first_movs = first_movs[
    (first_movs[RT_COL] >= 0.02) & (first_movs[RT_COL] < 1.0)
].copy()

print(f"Trials after filter: {len(first_movs)}")

In [ ]:
x   = first_movs["trial"].to_numpy()
rt  = first_movs[RT_COL].to_numpy()

ma_50  = pd.Series(rt).rolling(window=50,  center=True, min_periods=1).mean().to_numpy()
ma_100 = pd.Series(rt).rolling(window=100, center=True, min_periods=1).mean().to_numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, rt, ".", alpha=0.2, color=PALETTE["neutral"], label="per trial")
ax.plot(x, ma_50,  linewidth=1.5, color=PALETTE["pos"],     label="50-trial MA")
ax.plot(x, ma_100, linewidth=2,   color=PALETTE["neg"],     label="100-trial MA")
ax.set_xlabel("Trial")
ax.set_ylabel("Movement latency from go (s)")
ax.set_title(f"Session: {example_session}")
ax.legend(frameon=False, fontsize=8)
style_ax(ax)
plt.tight_layout()
save_fig(fig, "single_session_rt_timecourse", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 4. Population RT timecourse

For each session: z-score RT within session → 100-trial MA → interpolate to
a 0–1 fractional session axis → grand mean ± SEM across sessions.

Using the 0–1 axis avoids session-duration confounds.

In [ ]:
N_BINS   = 100  # grid resolution for 0–1 fraction axis
MA_WINDOW = 100  # moving-average window (trials)
x_grid = np.linspace(0.0, 1.0, N_BINS)

all_sessions = all_tongue_movements["session"].unique()
all_traces_rt = []

for sess in all_sessions:
    sub = all_tongue_movements[all_tongue_movements["session"] == sess]

    first = (
        sub.sort_values(["trial", "start_time"])
        .groupby("trial", as_index=False)
        .first()
        .sort_values("trial")
        .dropna(subset=[RT_COL])
    )
    first = first[(first[RT_COL] >= 0.02) & (first[RT_COL] < 1.0)]
    if len(first) < 20:
        continue

    y = first[RT_COL].to_numpy()
    y_mean, y_std = np.nanmean(y), np.nanstd(y)
    if not np.isfinite(y_std) or y_std == 0:
        continue

    y_z = (y - y_mean) / y_std
    y_smooth = (
        pd.Series(y_z)
        .rolling(window=MA_WINDOW, center=True, min_periods=1)
        .mean()
        .to_numpy()
    )

    n = len(y_smooth)
    x_sess = np.linspace(0.0, 1.0, n)
    y_interp = np.interp(x_grid, x_sess, y_smooth)
    all_traces_rt.append(y_interp)

stacked_rt = np.vstack(all_traces_rt)  # (n_sessions, N_BINS)
print(f"Sessions included: {len(all_traces_rt)}")

grand_mean_rt = np.nanmean(stacked_rt, axis=0)
n_eff = np.sum(~np.isnan(stacked_rt), axis=0)
grand_sem_rt  = np.nanstd(stacked_rt, axis=0) / np.sqrt(np.maximum(n_eff, 1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_grid * 100, grand_mean_rt, linewidth=2, color=PALETTE["neutral"],
        label=f"Grand mean  (n={len(all_traces_rt)} sessions)")
ax.fill_between(
    x_grid * 100,
    grand_mean_rt - grand_sem_rt,
    grand_mean_rt + grand_sem_rt,
    alpha=0.3, color=PALETTE["neutral"], label="±SEM",
)
ax.axhline(0, linestyle=":", linewidth=0.8, color="gray")
ax.set_xlabel("Session progress (%)")
ax.set_ylabel("Z-scored RT\n(100-trial MA)")
ax.set_title("Population RT timecourse — first movement per trial")
ax.legend(frameon=False, fontsize=9)
style_ax(ax)
plt.tight_layout()
save_fig(fig, "population_rt_timecourse", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 5. Vigor timecourse: first-movement vs all-movements

Compare session-normalized mean-velocity timecourse for:
- First movement of each trial (one data point per trial)
- All movements (more data, but potentially confounded by lick number)

In [ ]:
def _session_timecourse(
    df: pd.DataFrame,
    value_col: str,
    mode: str = "first",
    ma_window: int = 100,
    n_bins: int = 100,
):
    """
    Build population z-scored vigor timecourse.

    Parameters
    ----------
    df : DataFrame
        all_tongue_movements with 'session', 'trial', 'start_time', and value_col.
    value_col : str
        Column to analyze (e.g. 'mean_velocity', 'movement_latency_from_go').
    mode : str
        'first' — first movement per trial; 'all' — all movements.
    ma_window : int
        Moving-average window in movements/trials.
    n_bins : int
        Grid resolution for 0–1 session-fraction axis.

    Returns
    -------
    x_grid : ndarray  (0–1)
    grand_mean : ndarray
    grand_sem : ndarray
    n_sessions : int
    """
    x_grid = np.linspace(0.0, 1.0, n_bins)
    traces = []

    for sess, grp in df.groupby("session"):
        if "trial" in grp.columns:
            grp = grp[grp["trial"].notna()]

        if mode == "first":
            sub = (
                grp.sort_values(["trial", "start_time"])
                .groupby("trial", as_index=False)
                .first()
                .sort_values("trial")
            )
        else:
            sub = grp.sort_values(["trial", "start_time"])

        sub = sub.dropna(subset=[value_col])
        if len(sub) < 20:
            continue

        y = sub[value_col].to_numpy()
        y_mean, y_std = np.nanmean(y), np.nanstd(y)
        if not np.isfinite(y_std) or y_std == 0:
            continue

        y_z = (y - y_mean) / y_std
        y_smooth = (
            pd.Series(y_z)
            .rolling(window=ma_window, center=True, min_periods=1)
            .mean()
            .to_numpy()
        )

        n = len(y_smooth)
        x_sess = np.linspace(0.0, 1.0, n)
        y_interp = np.interp(x_grid, x_sess, y_smooth)
        traces.append(y_interp)

    if not traces:
        raise RuntimeError(f"No valid sessions for mode={mode!r}")

    stacked = np.vstack(traces)
    grand_mean = np.nanmean(stacked, axis=0)
    n_eff = np.sum(~np.isnan(stacked), axis=0)
    grand_sem = np.nanstd(stacked, axis=0) / np.sqrt(np.maximum(n_eff, 1))
    return x_grid, grand_mean, grand_sem, len(traces)

In [ ]:
# Build timecourse for both modes
modes = {
    "first": "First movement per trial",
    "all":   "All movements",
}

tc_results = {}
for mode, label in modes.items():
    x_g, gm, gs, n_sess = _session_timecourse(
        all_tongue_movements, VIGOR_COL, mode=mode,
    )
    tc_results[mode] = (x_g, gm, gs, n_sess, label)
    print(f"{label}: {n_sess} sessions")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (mode, (x_g, gm, gs, n_sess, label)) in zip(axes, tc_results.items()):
    ax.plot(x_g * 100, gm, linewidth=2, color=PALETTE["neutral"],
            label=f"Grand mean  (n={n_sess})")
    ax.fill_between(x_g * 100, gm - gs, gm + gs,
                    alpha=0.3, color=PALETTE["neutral"], label="±SEM")
    ax.axhline(0, linestyle=":", linewidth=0.8, color="gray")
    ax.set_xlabel("Session progress (%)")
    ax.set_title(label)
    ax.legend(frameon=False, fontsize=8)
    style_ax(ax)

axes[0].set_ylabel(f"Z-scored {VIGOR_COL.replace('_',' ')}\n(100-pt MA)")
fig.suptitle("Vigor timecourse: first-movement vs all-movements")
plt.tight_layout()
save_fig(fig, "vigor_timecourse_modes", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 6. Post-rewarded vs post-unrewarded trials

Is first-movement vigor higher on trials following a reward?

For each session: split first-movements by whether the *previous* trial was rewarded.
Z-score using session-level mean/std → 100-trial MA → interpolate to 0–1 axis.

In [ ]:
# Check which reward column is available
rew_candidates = ["rewarded", "earned_reward", "reward"]
REW_COL = next((c for c in rew_candidates if c in all_tongue_movements.columns), None)
print(f"Reward column: {REW_COL}")
if REW_COL is not None:
    print(all_tongue_movements[REW_COL].value_counts())

In [ ]:
def _interp_to_grid(time_list, trace_list, n_bins=100):
    """Interpolate per-session traces to a common 0–1 grid, returning grand mean ± SEM."""
    x_grid = np.linspace(0.0, 1.0, n_bins)
    stacked = np.full((len(trace_list), n_bins), np.nan)

    for i, (t, y) in enumerate(zip(time_list, trace_list)):
        if len(t) < 2:
            continue
        order   = np.argsort(t)
        t_s     = t[order]
        y_s     = y[order]
        y_interp = np.interp(x_grid, t_s, y_s)
        valid    = x_grid <= t_s[-1]
        y_interp[~valid] = np.nan
        stacked[i, :] = y_interp

    grand_mean = np.nanmean(stacked, axis=0)
    n_eff      = np.sum(~np.isnan(stacked), axis=0)
    grand_sem  = np.nanstd(stacked, axis=0) / np.sqrt(np.maximum(n_eff, 1))
    return x_grid, grand_mean, grand_sem

In [ ]:
if REW_COL is None:
    print("No reward column found — skipping post-rewarded analysis")
else:
    time_rew, trace_rew     = [], []
    time_unrew, trace_unrew = [], []

    for sess, grp in all_tongue_movements.groupby("session"):
        grp = grp[grp["trial"].notna()]

        first = (
            grp.sort_values(["trial", "start_time"])
            .groupby("trial", as_index=False)
            .first()
            .sort_values("trial")
        )

        if VIGOR_COL not in first.columns or REW_COL not in first.columns:
            continue

        first = first.dropna(subset=[VIGOR_COL, "start_time", REW_COL])
        if len(first) < 20:
            continue

        # Label each trial by whether the *previous* trial was rewarded
        first = first.sort_values("trial")
        first["prev_rewarded"] = first[REW_COL].shift(1)
        first = first.dropna(subset=["prev_rewarded"])
        if first.empty:
            continue

        # Z-score using session-level mean/std from ALL first movements
        s_mean = first[VIGOR_COL].mean()
        s_std  = first[VIGOR_COL].std()
        if not np.isfinite(s_std) or s_std == 0:
            continue

        t0 = first["start_time"].min()
        t1 = first["start_time"].max()
        if t1 <= t0:
            continue

        for is_rew, t_list, y_list in [
            (True,  time_rew,   trace_rew),
            (False, time_unrew, trace_unrew),
        ]:
            sub = first[first["prev_rewarded"] == is_rew]
            if sub.empty:
                continue

            t_norm = (sub["start_time"].to_numpy() - t0) / (t1 - t0)
            y_z    = (sub[VIGOR_COL].to_numpy() - s_mean) / s_std
            y_smooth = (
                pd.Series(y_z)
                .rolling(window=100, center=True, min_periods=1)
                .mean()
                .to_numpy()
            )
            t_list.append(t_norm)
            y_list.append(y_smooth)

    print(f"Post-rewarded sessions:   {len(time_rew)}")
    print(f"Post-unrewarded sessions: {len(time_unrew)}")

In [ ]:
if REW_COL is not None and time_rew and time_unrew:
    x_g, mean_rew,   sem_rew   = _interp_to_grid(time_rew,   trace_rew)
    _,   mean_unrew, sem_unrew = _interp_to_grid(time_unrew, trace_unrew)

    x_pct = x_g * 100

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x_pct, mean_rew, linewidth=2, color=PALETTE["pos"], label="After rewarded")
    ax.fill_between(x_pct, mean_rew - sem_rew, mean_rew + sem_rew,
                    alpha=0.25, color=PALETTE["pos"])
    ax.plot(x_pct, mean_unrew, linewidth=2, color=PALETTE["neg"], label="After unrewarded")
    ax.fill_between(x_pct, mean_unrew - sem_unrew, mean_unrew + sem_unrew,
                    alpha=0.25, color=PALETTE["neg"])
    ax.axhline(0, linestyle=":", linewidth=0.8, color="gray")
    ax.set_xlabel("Session progress (%)")
    ax.set_ylabel(f"Z-scored {VIGOR_COL.replace('_',' ')}\n(first movement, 100-trial MA)")
    ax.set_title("Vigor following rewarded vs unrewarded trials")
    ax.legend(frameon=False, fontsize=9)
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "vigor_post_reward_conditioning", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

## 7. Multiple kinematic features across the session

Run `_session_timecourse` for several kinematics features and compare slopes
to identify which features show the strongest slow modulation.

In [ ]:
# Kinematic features to examine
KIN_FEATURES = {
    "mean_velocity":          "Mean velocity",
    "movement_latency_from_go": "RT (latency from go)",
    "endpoint_y":             "Endpoint Y",
    "out_duration":           "Out-phase duration",
    "peak_velocity":          "Peak velocity",
}

# Keep only features present in the dataset
available_features = {
    k: v for k, v in KIN_FEATURES.items()
    if k in all_tongue_movements.columns
}
print("Available features:", list(available_features.keys()))

feature_tc = {}
for feat, label in available_features.items():
    try:
        x_g, gm, gs, n = _session_timecourse(
            all_tongue_movements, feat, mode="first",
        )
        feature_tc[feat] = (x_g, gm, gs, n, label)
        print(f"  {label}: {n} sessions OK")
    except Exception as e:
        print(f"  {label}: FAILED — {e}")

In [ ]:
if feature_tc:
    n_feat = len(feature_tc)
    ncols = min(n_feat, 3)
    nrows = (n_feat + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5 * ncols, 4 * nrows),
                             squeeze=False)
    ax_flat = [ax for row in axes for ax in row]

    for ax, (feat, (x_g, gm, gs, n_sess, label)) in zip(ax_flat, feature_tc.items()):
        ax.plot(x_g * 100, gm, linewidth=2, color=PALETTE["neutral"],
                label=f"n={n_sess}")
        ax.fill_between(x_g * 100, gm - gs, gm + gs,
                        alpha=0.3, color=PALETTE["neutral"])
        ax.axhline(0, linestyle=":", linewidth=0.8, color="gray")
        ax.set_xlabel("Session progress (%)")
        ax.set_ylabel("Z-score (100-trial MA)")
        ax.set_title(label)
        ax.legend(frameon=False, fontsize=8)
        style_ax(ax)

    for ax in ax_flat[len(feature_tc):]:
        ax.set_visible(False)

    fig.suptitle("Population timecourse — kinematic features (first movement per trial)")
    plt.tight_layout()
    save_fig(fig, "kin_feature_timecourses", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()